# Category-reranked object RAG: design and plan

Question + category → mapped active documents across categories → eligible table/outer-row vectors → cosine candidate pool → category score reranking → exact cited evidence → Ollama code answer.

Retrieve up to four times `top_k` cosine matches, then rank by `score = (1 - cosine_distance) + CATEGORY_SCORE_BOOST * category_match`. The default bonus is `0.15`; configure `OBJECT_RAG_CATEGORY_BOOST` or `CATEGORY_SCORE_BOOST` below. Matching manual categories receive the bonus; other mapped categories remain eligible. Keep the raw distance alongside the similarity, applied bonus and final score for inspection. The constructor callback applies this ranking to direct `app.retrieve` and `app.answer` calls as well as the coverage suites. Saved outputs reflect earlier executions; restart the kernel, then rerun the configuration, service and selected retrieval/answer cells to refresh them.

Reuse the existing index without ingestion or checkpoint changes. Match the combined and V4.0/V5.0 workbook mappings through explicit corpus-root aliases, retain corpus and revision labels, and report missing coverage. Traverse row-group wrappers for outer rows; exclude cells and deeper rows from vector candidates. Validate identity, embedding configuration and dimensions; query eligible IDs before ranking.

Implementation: configure paths and services; load categories/mappings; open the catalog read-only and existing collection; inspect coverage; retrieve and inspect exact evidence; generate version-specific fenced snippets with citations; close resources. Defaults: supplied index, top-k 8, 24,000 context characters, Ollama RAG model falling back to the summary model. Complete blocks exceeding the budget are reported as omitted. Unsupported coding requests must receive an insufficient-evidence answer.

Validation: offline fake-provider and ephemeral-index tests, notebook validation and cell compilation, and read-only real catalog coverage inspection. No corpus rebuilds, summary requests or checkpoint resets.


Every answer includes a separate source-attributed summary of the retrieved objects, regardless of whether the coding request is supported.

## Application configuration

Reusable retrieval logic lives in [01.rag.object.ipynb](01.rag.object.ipynb), exported as `ribosome.rag.object`. Configuration and client lifecycle stay here.

In [1]:
# | notest
import math
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from ribosome.rag.object import ObjectRAG

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
load_dotenv(PROJ_ROOT / ".env", override=False)
# INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261022-152558-151254"))
INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261026-090625-217683"))
if not INDEX_DIR.is_absolute(): INDEX_DIR = PROJ_ROOT / INDEX_DIR
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
HOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
REQUEST_TIMEOUT = 300
MAX_OUTPUT_TOKENS = 2048
CATEGORY_SCORE_BOOST = float(os.getenv("OBJECT_RAG_CATEGORY_BOOST", "0.15"))
if not math.isfinite(CATEGORY_SCORE_BOOST) or CATEGORY_SCORE_BOOST < 0:
    raise ValueError("OBJECT_RAG_CATEGORY_BOOST must be finite and nonnegative")

def rerank_by_category(hits, category):
    """Boost matching manual labels while retaining semantic similarity."""
    scored = []
    for hit in hits:
        similarity = 1.0 - hit["distance"]
        category_match = hit["manual"]["category"] == category
        boost = CATEGORY_SCORE_BOOST if category_match else 0.0
        scored.append(dict(hit, similarity=similarity, category_match=category_match,
                           category_boost=boost, score=similarity + boost))
    return sorted(scored, key=lambda hit: (-hit["score"], hit["distance"], hit["record"]["id"]))



## Open existing index and services
Run explicitly; no embedding or generation occurs in this cell.

In [2]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / 'catalog.sqlite3').is_file() or not (INDEX_DIR / 'chroma').is_dir():
    raise FileNotFoundError(INDEX_DIR)
headers = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
embedding_client = AsyncClient(host=HOSTS[0], headers=headers, timeout=REQUEST_TIMEOUT)
generation_client = AsyncClient(host=RAG_HOST, headers=headers, timeout=REQUEST_TIMEOUT)
collection = chromadb.PersistentClient(path=str(INDEX_DIR/'chroma')).get_collection('object_summaries', embedding_function=None)
app = ObjectRAG(
    INDEX_DIR, collection, embedding_client, generation_client,
    mapping_dir=PROJ_ROOT / 'assets/coding_categories',
    embed_model=EMBED_MODEL,
    embedding_config=dict(provider='ollama-pool', hosts=HOSTS),
    rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
    max_output_tokens=MAX_OUTPUT_TOKENS, rerank=rerank_by_category,
)
async def retrieve(question, category, *, top_k=8):
    return await app.retrieve(question, category, top_k=top_k)
async def answer(question, category, *, top_k=8):
    return await app.answer(question, category, top_k=top_k)


## Category coverage

This suite has one prompt for each of the **17 application categories other than
`general_operation`**. Questions may use either mapped manual revision, with the
answer identifying its source. The dedicated manual-version suites and expanded CSV evaluation now live in
[06.test.rag.object.ipynb](06.test.rag.object.ipynb).

Set `COVERAGE_CASE_IDS` to selected category case IDs, or leave it `None` for all
17 prompts. The following two sections retrieve evidence and generate answers for
this suite only. Defining or displaying prompts makes no model requests. Shared
helpers also serve the separate manual suite, which has its own selection and
result dictionaries.


In [3]:
# | notest
COVERAGE_CASES = [{'id': 'category_adhesive_dispensing',
  'category': 'adhesive_dispensing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。',
  'references': []},
 {'id': 'category_arc_welding',
  'category': 'arc_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用ARCON开始弧焊、沿焊缝运动并用ARCOFF结束焊接？请说明起弧文件与焊接文件的区别。',
  'references': []},
 {'id': 'category_fds',
  'category': 'fds',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何按FDS手册配置主机工作参数和IO，并组织一次FDS作业的启动、完成状态确认及后续动作？',
  'references': []},
 {'id': 'category_grinding',
  'category': 'grinding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何首次配置打磨力控制器并完成标定，再设置打磨力、位移及出入刀监控，组织一次打磨作业？',
  'references': []},
 {'id': 'category_laser_welding',
  'category': 'laser_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '激光焊作业需要哪些激光器、通讯和IO配置？如何按手册组织激光开启、焊接运动和激光关闭的程序？',
  'references': []},
 {'id': 'category_load_analysis',
  'category': 'load_analysis',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用负载分析软件时，如何确认工具数模坐标系与机器人法兰坐标系一致，并查看和解释负载分析结果？只需说明操作流程，不要虚构控制指令。',
  'references': []},
 {'id': 'category_material_handling',
  'category': 'material_handling',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置搬运应用的抓手、工件传感器和电磁阀，并组织抓取工件、确认抓取状态、搬运后释放的作业？',
  'references': []},
 {'id': 'category_offline_interface',
  'category': 'offline_interface',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '使用新松机器人离线接口库时，上位机初始化、建立网络连接、发送运动或下载作业、断开并释放资源的正确调用顺序是什么？请选择一种手册支持的语言举例。',
  'references': []},
 {'id': 'category_palletizing',
  'category': 'palletizing',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何从开启码垛应用、PLC通讯准备、工具与用户坐标系设置开始，配置产品和码垛工艺，再组织一次码垛作业？',
  'references': []},
 {'id': 'category_pneumatic_spot_welding',
  'category': 'pneumatic_spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置气动点焊的设备交互IO、焊接压力和焊点参数，并组织加压、焊接、完成确认及松开焊钳的流程？',
  'references': []},
 {'id': 'category_spot_welding',
  'category': 'spot_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '伺服点焊作业前如何完成焊钳轴校零、压力标定和焊点参数配置，并组织一次加压、焊接及打开焊钳的流程？',
  'references': []},
 {'id': 'category_spr_riveting',
  'category': 'spr_riveting',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置SPR铆机的工作模式、IO及通讯，并按基本作业流程完成一次铆接和防漏铆确认？',
  'references': []},
 {'id': 'category_stud_welding',
  'category': 'stud_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置螺柱焊焊机的工作参数、IO及通讯，并按基本作业流程完成一次焊接及防漏焊确认？',
  'references': []},
 {'id': 'category_teaching_free_welding',
  'category': 'teaching_free_welding',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '免示教焊接软件如何配置机器人与视觉设备、创建工件、设置扫描区域和焊缝规则，再规划并执行焊接任务？请区分服务端配置和客户端操作。',
  'references': []},
 {'id': 'category_tool_quick_change',
  'category': 'tool_quick_change',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何配置快换停靠站、快换盘和IO，并组织放下当前工具、确认解锁、拾取目标工具和确认锁紧的基本作业流程？',
  'references': []},
 {'id': 'category_tracking',
  'category': 'tracking',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何使用FL_GETV读取第一路传送带的实时速度，并在跟随作业中使用该速度？请说明R变量、通路参数及单位。',
  'references': []},
 {'id': 'category_vision',
  'category': 'vision',
  'kind': 'category',
  'profile': '',
  'revision': '',
  'section': '',
  'section_title': '',
  'anchors': [],
  'question': '如何给相机发送拍照指令、获取测量结果并计算位姿偏移？请按同一手册版本说明视觉作业参数及指令调用顺序。',
  'references': []}]


In [4]:
# | notest
import html
import re
from copy import deepcopy

def coverage_plain(text):
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', ' ', text)).split())

def coverage_heading_title(heading):
    title = re.sub(r'^\s*\d+(?:\.\d+)*\s+', '', heading.strip('* '))
    return re.sub(r'\s+', '', title).translate(str.maketrans({'—': '-', '–': '-', '－': '-'}))

def coverage_heading_path_matches(expected_path, headings):
    remaining = iter(coverage_heading_title(heading) for heading in headings)
    return all(any(title == coverage_heading_title(expected) for title in remaining)
               for expected in expected_path)

def coverage_case_metrics(case, result, error=''):
    hits = result.get('hits', [])
    # Version-specific cases must find their own corpus and manual, even though
    # production retrieval is category-reranked and may return both revisions.
    matching = [hit for hit in hits if not case['profile'] or (
        hit['manual']['profile'] == case['profile'] and
        Path(hit['record']['source_file']).stem == Path(case['references'][0]['docx']).stem)]
    headings = [heading.split(maxsplit=1)[0] for hit in matching
                for heading in hit['record'].get('section_headings', []) if heading.strip()]
    if case.get('heading_path'):
        section_hit = any(coverage_heading_path_matches(
            case['heading_path'], hit['record'].get('section_headings', [])) for hit in matching)
    else:
        section_hit = case['section'] in headings if case['section'] else None
    source_text = '\n'.join(coverage_plain(hit['record']['content']) for hit in matching)
    anchors_found = all(re.search(r'(?<![A-Za-z0-9_])'+re.escape(term)+r'(?![A-Za-z0-9_])',
                                 source_text, flags=re.IGNORECASE) for term in case['anchors'])
    return dict(case_id=case['id'], category=case['category'], kind=case['kind'],
                expected_revision=case['revision'], expected_section=case['section'],
                hits=len(hits), expected_manual_hit=bool(matching) if case['profile'] else None,
                expected_section_hit=section_hit,
                anchors_in_matching_hits=anchors_found if case['anchors'] else None,
                context_chars=len(result.get('context', '')), omissions=len(result.get('omissions', [])),
                missing_manuals=len(result.get('coverage', {}).get('missing', [])), error=error)

def show_coverage_retrieval(case_id, *, cases=None, results=None, errors=None):
    cases = coverage_cases if cases is None else cases
    results = coverage_retrieval_results if results is None else results
    errors = coverage_retrieval_errors if errors is None else errors
    case = next(case for case in cases if case['id'] == case_id)
    display(Markdown(f"### {case_id}\n\n{case['question']}"))
    if case_id in errors:
        print(errors[case_id])
        return
    result = results[case_id]
    display(pd.DataFrame([dict(score=hit.get('score'), similarity=hit.get('similarity'),
                               category_match=hit.get('category_match'),
                               category_boost=hit.get('category_boost'),
                               distance=hit['distance'], manual=hit['manual'],
                               lines=(hit['record']['attributes'].get('line_start'),
                                      hit['record']['attributes'].get('line_end')),
                               summary=hit['record'].get('summary', ''),
                               source=hit['record']['source_file']) for hit in result['hits']]))
    print(result['context'])
    print('Omitted:', result['omissions'])
    # Reference excerpts remain reviewer data and are never added to the prompt.
    if case['references']:
        display(pd.DataFrame(case['references']))

class CoverageAnswerReplay(ObjectRAG):
    """Run the production answer method on the exact retrieved test evidence."""
    def __init__(self, application, evidence, *, top_k=8):
        self.generator = application.generator
        self.rag_model = application.rag_model
        self.max_output_tokens = application.max_output_tokens
        self.evidence = evidence
        self.top_k = top_k
        self.last_result = None

    async def retrieve(self, question, category, *, top_k=8):
        if (question, category, top_k) != (
            self.evidence['question'], self.evidence['category'], self.top_k
        ):
            raise ValueError('Coverage replay requires the same question, category and top-k')
        self.last_result = deepcopy(self.evidence)
        return self.last_result

async def run_coverage_retrievals(application, cases, *, top_k, results, errors):
    """Retrieve into this suite's dictionaries; retain completed cases on interruption."""
    for number, case in enumerate(cases, 1):
        print(f"[{number}/{len(cases)}] Retrieve {case['id']}", flush=True)
        try:
            results[case['id']] = await application.retrieve(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case['id']] = f'{type(error).__name__}: {error}'
    return pd.DataFrame([
        coverage_case_metrics(case, results.get(case['id'], {}),
                              errors.get(case['id'], ''))
        for case in cases])

async def run_coverage_answers(application, cases, *, top_k, retrieval_results,
                               retrieval_errors, results, errors):
    """Answer from this suite's frozen evidence; never make another retrieval request."""
    for number, case in enumerate(cases, 1):
        case_id = case['id']
        display(Markdown(f"## [{number}/{len(cases)}] {case_id}\n\n{case['question']}"))
        if case_id not in retrieval_results:
            errors[case_id] = retrieval_errors.get(case_id, 'Retrieve this case first')
            display(Markdown('Retrieval failed: ' + errors[case_id]))
            continue
        replay = CoverageAnswerReplay(application, retrieval_results[case_id], top_k=top_k)
        try:
            results[case_id] = await replay.answer(
                case['question'], case['category'], top_k=top_k)
        except Exception as error:
            errors[case_id] = f'{type(error).__name__}: {error}'
            results[case_id] = replay.last_result or {}
        finally:
            if case_id not in results and replay.last_result is not None:
                results[case_id] = replay.last_result
                errors[case_id] = 'Generation interrupted before completion'
            case_result = results.get(case_id, {})
            if 'retrieved_summary' in case_result:
                display(Markdown('### Retrieved content summary\n\n' + case_result['retrieved_summary']))
        if case_id in errors:
            display(Markdown('Generation failed: ' + errors[case_id]))
        else:
            display(Markdown('### Coding or diagnostic answer\n\n' + case_result['answer']))
    return pd.DataFrame([
        dict(**coverage_case_metrics(case, results.get(case['id'], {}),
                                     errors.get(case['id'], '')),
             example_status=results.get(case['id'], {}).get('example_status', 'not generated'))
        for case in cases])


In [5]:
# | notest
display(pd.DataFrame([(category, name) for category, name in app.categories.items()
                      if category != 'general_operation'], columns=['category', 'chinese_name']))
assert {case['category'] for case in COVERAGE_CASES} == set(app.categories) - {'general_operation'}
COVERAGE_TOP_K = 8
COVERAGE_CASE_IDS = None  # Example: ['category_arc_welding', 'category_vision']
known_case_ids = {case['id'] for case in COVERAGE_CASES}
if COVERAGE_CASE_IDS is not None:
    if not COVERAGE_CASE_IDS or set(COVERAGE_CASE_IDS) - known_case_ids:
        raise ValueError('Select at least one known coverage case ID')
coverage_cases = [case for case in COVERAGE_CASES
                  if COVERAGE_CASE_IDS is None or case['id'] in COVERAGE_CASE_IDS]
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(pd.DataFrame(coverage_cases)[
        ['id', 'category', 'question']])
category_coverage_rows = []
for category in dict.fromkeys(case['category'] for case in coverage_cases):
    mapped_coverage = app.snapshot(category)[2]
    category_coverage_rows.append(dict(category=category, indexed=mapped_coverage['indexed'],
                                       missing=mapped_coverage['missing']))
display(pd.DataFrame(category_coverage_rows))


,category,chinese_name
0,adhesive_dispensing,涂胶
1,arc_welding,弧焊
2,fds,FDS
3,grinding,打磨
4,laser_welding,激光焊
5,load_analysis,负载分析
6,material_handling,搬运
7,offline_interface,离线接口
8,palletizing,码垛
9,pneumatic_spot_welding,气动点焊


,id,category,question
0,category_adhesive_dispensing,adhesive_dispensing,如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。
1,category_arc_welding,arc_welding,如何使用ARCON开始弧焊、沿焊缝运动并用ARCOFF结束焊接？请说明起弧文件与焊接文件的区别。
2,category_fds,fds,如何按FDS手册配置主机工作参数和IO，并组织一次FDS作业的启动、完成状态确认及后续动作？
3,category_grinding,grinding,如何首次配置打磨力控制器并完成标定，再设置打磨力、位移及出入刀监控，组织一次打磨作业？
4,category_laser_welding,laser_welding,激光焊作业需要哪些激光器、通讯和IO配置？如何按手册组织激光开启、焊接运动和激光关闭的程序？
5,category_load_analysis,load_analysis,使用负载分析软件时，如何确认工具数模坐标系与机器人法兰坐标系一致，并查看和解释负载分析结果？只需说明操作流程，不要虚构控制指令。
6,category_material_handling,material_handling,如何配置搬运应用的抓手、工件传感器和电磁阀，并组织抓取工件、确认抓取状态、搬运后释放的作业？
7,category_offline_interface,offline_interface,使用新松机器人离线接口库时，上位机初始化、建立网络连接、发送运动或下载作业、断开并释放资源的正确调用顺序是什么？请选择一种手册支持的语言举例。
8,category_palletizing,palletizing,如何从开启码垛应用、PLC通讯准备、工具与用户坐标系设置开始，配置产品和码垛工艺，再组织一次码垛作业？
9,category_pneumatic_spot_welding,pneumatic_spot_welding,如何配置气动点焊的设备交互IO、焊接压力和焊点参数，并组织加压、焊接、完成确认及松开焊钳的流程？


,category,indexed,missing
0,adhesive_dispensing,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人...,[]
1,arc_welding,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人...,[]
2,fds,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322016《新松机器人...,[]
3,grinding,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322006《新松机器人...,[]
4,laser_welding,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322004《新松机器人...,[]
5,load_analysis,[{'key': '06 产品手册/软件类/手册/V3.0版本/SX322010《新松机器人...,[]
6,material_handling,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322011《新松机器人...,[]
7,offline_interface,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322009《新松机器人...,[]
8,palletizing,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322002《新松机器人...,[]
9,pneumatic_spot_welding,[{'key': '06 产品手册/软件类/手册/V5.0版本/SX322018《新松机器人...,[]


In [6]:
# | notest
INSPECT_CASE_ID = coverage_cases[0]['id']  # Change to any selected ID to inspect its exact evidence.


## Retrieve and inspect exact evidence

Run the selected category prompts through the existing category-reranked retriever.
Results are retained by case ID in `coverage_retrieval_results`, with per-case
failures in `coverage_retrieval_errors`; completed results remain inspectable after
interruption. Use `show_coverage_retrieval(case_id)` for any selected category case
to inspect the matched objects, raw context and omissions.

The category table reports hit count, context size, omitted objects, missing mapped
manuals and errors. These are retrieval diagnostics, not answer-correctness scores.
Version, subsection and alarm-code checks belong to the dedicated manual suite
below. The original production candidate restrictions still apply.


In [7]:
# | notest
coverage_retrieval_results, coverage_retrieval_errors = {}, {}
coverage_retrieval_metrics = await run_coverage_retrievals(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    results=coverage_retrieval_results, errors=coverage_retrieval_errors)
display(coverage_retrieval_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'error']])
show_coverage_retrieval(INSPECT_CASE_ID)


[1/17] Retrieve category_adhesive_dispensing
[2/17] Retrieve category_arc_welding
[3/17] Retrieve category_fds
[4/17] Retrieve category_grinding
[5/17] Retrieve category_laser_welding
[6/17] Retrieve category_load_analysis
[7/17] Retrieve category_material_handling
[8/17] Retrieve category_offline_interface
[9/17] Retrieve category_palletizing
[10/17] Retrieve category_pneumatic_spot_welding
[11/17] Retrieve category_spot_welding
[12/17] Retrieve category_spr_riveting
[13/17] Retrieve category_stud_welding
[14/17] Retrieve category_teaching_free_welding
[15/17] Retrieve category_tool_quick_change
[16/17] Retrieve category_tracking
[17/17] Retrieve category_vision


,case_id,category,hits,context_chars,omissions,missing_manuals,error
0,category_adhesive_dispensing,adhesive_dispensing,8,12971,0,0,
1,category_arc_welding,arc_welding,8,17215,0,0,
2,category_fds,fds,8,6403,0,0,
3,category_grinding,grinding,8,10491,0,0,
4,category_laser_welding,laser_welding,8,11921,0,0,
5,category_load_analysis,load_analysis,8,4460,0,0,
6,category_material_handling,material_handling,8,9870,0,0,
7,category_offline_interface,offline_interface,8,18685,0,0,
8,category_palletizing,palletizing,8,8970,0,0,
9,category_pneumatic_spot_welding,pneumatic_spot_welding,8,9161,0,0,


### category_adhesive_dispensing

如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。

,score,similarity,category_match,category_boost,distance,manual,lines,summary,source
0,0.859229,0.709229,True,0.15,0.290771,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(701, 778)",该表展示了一段机器人涂胶程序的完整流程，按行顺序定义了**坐标系/负载配置**、**运动路径...,/d/devel/rag/ribosome/assets/code-docs.5.0/.md...
1,0.837284,0.687284,True,0.15,0.312716,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(388, 392)",涂胶结束指令集功能：涂胶结束，进行测量并根据参数是否进行填充,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
2,0.815880,0.665880,True,0.15,0.334120,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(784, 877)",该表格展示了一段机器人涂胶应用的 PLC 控制逻辑，主要流程为：**前置条件校验 -> 系统...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
3,0.807394,0.657394,True,0.15,0.342606,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(384, 387)",涂胶初始化指令集举例：A08_GLUE GlueInit Gun=1 ProgNo=1 Pa...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
4,0.803008,0.653008,True,0.15,0.346992,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(469, 472)",在“设置涂胶系统指令集”模块下，该行为示例条目，展示了设置涂胶系统的指令格式：`A08_GL...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...
5,0.802646,0.652646,True,0.15,0.347354,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(784, 877)",该表格展示了一段用于控制胶涂覆系统的梯形图或逻辑指令序列，主要执行流程包括：\n\n1. ...,/d/devel/rag/ribosome/assets/code-docs.5.0/.md...
6,0.783083,0.633083,True,0.15,0.366917,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(298, 350)",该表格定义了三种**GLUEL** 涂胶控制指令，分别对应不同的胶枪状态操作。各指令结构相似...,/d/devel/rag/ribosome/assets/code-docs.5.0/.md...
7,0.782231,0.632231,True,0.15,0.367769,{'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂...,"(376, 379)",涂胶初始化指令集：格式为 `A08_GLUE GlueInit Gun= ProgNo= P...,/d/devel/rag/ribosome/assets/code-docs.4.0/.md...


[E1] {'key': '06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1)', 'profile': '5.0', 'revision': 'V5.0 / A-1', 'category': 'adhesive_dispensing'} | /d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md | lines 701-778 | 37c4f5108de3bb61e98ae85067745cd52c511dea5e3d0119a6a9619c1525326f
<table>
<tr>
<td><p>NOP   </p></td>
<td></td>
</tr>
<tr>
<td><p>SET UF   &#35;0</p></td>
<td><p>用户坐标系选择</p></td>
</tr>
<tr>
<td><p>SET TF   &#35;1</p></td>
<td><p>工具坐标系选择</p></td>
</tr>
<tr>
<td><p>PAYLOAD   &#35;1</p></td>
<td><p>负载&#45;抓手加车架</p></td>
</tr>
<tr>
<td><p>MOVJ	PR102 VJ=30&#46;0 ACC=30 CNT=0</p></td>
<td><p>冲锋位</p></td>
</tr>
<tr>
<td><p>MOVJ	P1 VJ=30&#46;0 ACC=100 CNT=1</p></td>
<td></td>
</tr>
<tr>
<td><p>;</p></td>
<td></td>
</tr>
<tr>
<td><p>A08&#95;GLUE GlueSetSysNum SysNum=1</p></td>
<td><p>设置涂胶系统</p></td>
</tr>
<tr>
<td><p>A08&#95;GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill

## Retrieved content summary and coding answer

Always display a source-attributed summary of the found objects, even when their content cannot support the requested code. The summary uses stored object summaries and remains available if generation fails. With no hits, explicitly report that no relevant indexed content was found.

The coding answer ends with a meaningful fenced code snippet for the selected category, integrating the instruction requested in the question. Its source citations, corpus/revision, and required placeholders appear directly before the code block. If the evidence cannot support such an example, the answer explains the missing information.

A rich example includes a concrete scenario, environment and initial values, expected outcome, and a final multi-statement code block with setup, the requested instruction, and follow-up. The shared RAG class validates structured generation and retries once if incomplete. If evidence cannot support the surrounding code, it explains the missing context. After updating the shared module, restart the kernel and rerun configuration and client setup to load the new implementation.

Each selected case reuses the **exact evidence from the preceding retrieval pass**
through the production answer method, avoiding a second embedding request or a
different retrieval result. It displays a retrieved-content summary and answer for
every case; generation failures retain the available summary. Completed answers
remain in `coverage_answer_results` if execution is interrupted.

The coding-answer validator may return `unsupported` for GUI or workflow
questions, which is separate from retrieval coverage. Review the explanation and
sources. A structurally `included` example still requires source checking.


In [8]:
# | notest
coverage_answer_results, coverage_answer_errors = {}, {}
coverage_answer_metrics = await run_coverage_answers(
    app, coverage_cases, top_k=COVERAGE_TOP_K,
    retrieval_results=coverage_retrieval_results, retrieval_errors=coverage_retrieval_errors,
    results=coverage_answer_results, errors=coverage_answer_errors)
display(coverage_answer_metrics[
    ['case_id', 'category', 'hits', 'context_chars', 'omissions', 'missing_manuals', 'example_status', 'error']])


## [1/17] category_adhesive_dispensing

如何按涂胶手册配置胶机并编写一次涂胶流程，包含开始出胶、沿轨迹涂胶和停止出胶？请说明工艺文件和IO的准备条件。

### Retrieved content summary

- 该表展示了一段机器人涂胶程序的完整流程，按行顺序定义了**坐标系/负载配置**、**运动路径**、**涂胶控制**及**程序终止**四个阶段：

1.  **初始化设置**：
    *   设置用户坐标系（`SET UF #0`）和工具坐标系（`SET TF #1`）。
    *   定义负载为“抓手加车架”（`PAYLOAD #1`）。

2.  **定位与准备**：
    *   快速移动至“冲锋位”（`PR102`）及过渡点 `P1`。
    *   初始化涂胶系统：设置系统编号为1（`GlueSetSysNum`），并对1号胶枪执行初始化，包括冲洗和填充（`GlueInit`）。

3.  **涂胶执行**：
    *   线性移动至涂胶起点 `P4`。
    *   **开胶**：在 `P1` 执行 `GLUEL` 指令，开启1号胶枪，进行校准和涂胶（持续20ms）。
    *   **关胶**：执行 `GLUEL` 指令，关闭1号胶枪（停止时间20ms）。
    *   线性返回至 `P4`。

4.  **收尾与结束**：
    *   结束涂胶循环（`GlueEnd`）。
    *   执行胶枪填充检查（`GlueFillCheck`）。
    *   快速移动至点 `P2`，再返回“冲锋位”（`PR102`）。
    *   程序返回（`RET`）并结束（`END`）。

**核心结构**：表格共19行，前两列分别为**指令代码**和**功能说明/注释**，无合并单元格，逻辑清晰，涵盖了从系统配置、安全定位、胶枪初始化、精确涂胶动作到安全复位的全自动化涂胶工序。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 701–778; corpus 5.0, revision V5.0 / A-1; object `37c4f5108de3bb61e98ae85067745cd52c511dea5e3d0119a6a9619c1525326f`.

- 涂胶结束指令集功能：涂胶结束，进行测量并根据参数是否进行填充
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 388–392; corpus 4.0, revision V5.0 / A-1; object `ad90fdf2195c5a0665d2ed14a835fb0bc4d3e070ab3f50b3d900862f07b0f2ce`.

- 该表格展示了一段机器人涂胶应用的 PLC 控制逻辑，主要流程为：**前置条件校验 -> 系统初始化与硬件准备 -> 胶机配置与启动**。具体步骤如下：

1.  **模式与状态检查**：首先判断胶机是否处于模拟模式（`IN#38`）及机器人模式（`I498`），若不符合要求则跳转至结束标签 `L101`；随后等待进入自动模式（`IN#65`）。
2.  **系统接通与硬件触发**：置位系统接通（`OT#46`）和运行许可（`OT#40`）信号；触发加热器（`OT#47`）并等待其无故障（`IN#72`）、接通（`IN#68`）及就绪（`IN#69`）；触发泵（`OT#48`）并等待其接通（`IN#66`）。
3.  **胶路预处理与配置**：调用冲洗（`GLUEPURGE`）和填充（`GLUEFILL`指令集，并输出程序号（`OG#1`=I492）与参数号（`OG#2`=I493）至胶机，分别等待胶机反馈确认。
4.  **启动执行**：发出启动信号（`OT#33`）并等待涂胶准备就绪信号（`IN#48`），最后通过 `RET` 和 `END` 结束程序。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 784–877; corpus 4.0, revision V5.0 / A-1; object `8367b5b11b1bfc36e27dddffd13b8dfccc9b8c764063b98b40e1cabec938c0c7`.

- 涂胶初始化指令集举例：A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes 强制填充，冲洗
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 384–387; corpus 4.0, revision V5.0 / A-1; object `bc3b4dbcf1e29175d743d79585042e8b8627f02152de7818d9f7b32e22b106dd`.

- 在“设置涂胶系统指令集”模块下，该行为示例条目，展示了设置涂胶系统的指令格式：`A08_GLUE GlueSetSysNum SysNum=1涂胶系统`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 469–472; corpus 4.0, revision V5.0 / A-1; object `468d1b33a0f58e044ed6fcd4bb307145dc9f67bfc31b4d9f9f5f2e4294a16dd9`.

- 该表格展示了一段用于控制胶涂覆系统的梯形图或逻辑指令序列，主要执行流程包括：

1.  **模式检查与跳转**：首先检查胶机是否处于模拟模式（IN#38）或机器人是否处于非正常模式（I498），若满足条件则跳转至标签 L101 结束流程。
2.  **启动准备**：等待自动模式（IN#65），随后置位系统接通（OT#46）和运行许可（OT#40）信号。
3.  **硬件预启与状态确认**：脉冲触发加热器（OT#47）和泵（OT#48），并依次等待加热器/泵无故障、接通、准备就绪以及泵接通信号。
4.  **核心工艺执行**：调用冲洗（GLUEPURGE）和填充（GLUEFILL）指令集。
5.  **参数交互与启动**：向胶机输出程序号（OG#1/I492）和参数号（OG#2/I493）并等待反馈确认，最后置位启动信号（OT#33）并等待涂胶准备就绪信号（IN#48）。
6.  **流程结束**：跳转至标签 L101，执行返回（RET）和结束（END）指令。

整体逻辑为：在确认系统处于自动模式且安全无故障后，初始化硬件，执行胶水准备动作，同步程序参数，并最终启动涂胶流程。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 784–877; corpus 5.0, revision V5.0 / A-1; object `956004adc193d9a5394c2a5d13ce56f1f559bc365df480c9bb5dfb0aa5fecfee`.

- 该表格定义了三种**GLUEL** 涂胶控制指令，分别对应不同的胶枪状态操作。各指令结构相似，均包含基础运动参数（点位、速度、加速度、逼近值、工具/锁定配置）及特定的功能与时序参数。摘要如下：

1.  **开胶枪指令 (GLUEON)**
    *   **功能**：启动胶枪。
    *   **关键参数**：在基础参数基础上，包含 `GUN`（定量机）、`PARANo`（涂胶参数文件号）及 `TIME/DIS`（提前开胶时间/距离，负值提前，正值滞后）。
    *   **示例**：`GLUEL P1 VL=300 ... GLUEON GUN=1 ... TIME=20`

2.  **关胶枪指令 (GLUEOFF)**
    *   **功能**：停止胶枪。
    *   **关键参数**：结构较简，主要包含基础参数及 `TIME/DIS`（提前关胶时间/距离，负值提前，正值滞后）。
    *   **示例**：`GLUEL P1 VL=300 ... GLUEOFF TIME=20`

3.  **切换参数指令 (GLUESWITCH)**
    *   **功能**：切换涂胶参数。
    *   **关键参数**：包含基础参数、`PARANo`（新参数文件号）及 `TIME/DIS`（提前切换时间/距离，负值提前，正值滞后）。
    *   **示例**：`GLUEL P1 VL=300 ... GLUESWITCH ... PARANo=2 TIME=10`

**通用说明**：
*   所有指令均通过 `VL`（速度）、`ACC`（加速度）、`CNT`（逼近值）控制运动过程。
*   `TIME/DIS` 参数用于精确控制动作触发时机，支持基于时间或距离的提前/滞后调整。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 298–350; corpus 5.0, revision V5.0 / A-1; object `99f2a2d92483e7d2e609465d724648056157d64832e8c96af429534e9fd8f6c5`.

- 涂胶初始化指令集：格式为 `A08_GLUE GlueInit Gun= ProgNo= ParaNo= Purge=No/Yes/Auto Fill=No/Yes/ Auto`。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322012《新松机器人涂胶应用操作手册》(A-1)/SX322012《新松机器人涂胶应用操作手册》(A-1).md`, lines 376–379; corpus 4.0, revision V5.0 / A-1; object `846e243eb9a5817043700c17954e434f661719d847523c20a817c56b50c296c5`.

### Coding or diagnostic answer

按涂胶手册（SX322012 V5.0/A-1）配置胶机并编写一次涂胶流程需分三步：

一、工艺文件与IO准备条件（E3）：
- 确认胶机非模拟模式（IN#38≠ON）且机器人处于NORMAL模式（I498=0）；
- 等待自动模式信号 IN#65=ON；
- 置ON系统接通信号 OT#46 和运行许可信号 OT#40；
- 触发加热器（PULSE OT#47），等待加热器无故障 IN#72、接通 IN#68、就绪 IN#69；
- 触发泵（PULSE OT#48），等待泵接通 IN#66；
- 调用冲洗（CALL GLUEPURGE）和填充（CALL GLUEFILL）；
- 通过 OG#1/OG#2 输出程序号（I492）和参数号（I493）至胶机，并等待 IG#1/IG#2 反馈一致；
- 发出启动信号 OT#33=ON，等待涂胶准备就绪 IN#48=ON。

二、胶机配置（E1、E2、E5）：
- 用 A08_GLUE GlueSetSysNum SysNum=1 选择涂胶系统（存外部实型变量497）；
- 用 A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes 完成初始化，其中 ProgNo 存变量492、ParaNo 存变量493、Purge 存变量494、Fill 存变量495。

三、涂胶流程（E1、E7）：
- 开胶：GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo=1 TIME=20，沿点位P1线性运动同时开启1号胶枪，TIME=20表示提前20ms开胶；
- 沿轨迹涂胶：在GLUEON与GLUEOFF之间可插入多段MOVL/GLUEL指令完成轨迹；
- 关胶：GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEOFF TIME=20，到达终点后关闭胶枪；
- 结束：A08_GLUE GlueEnd Gun=1 Fill=No 进行测量，GlueFillCheck Gun=1 Check 检查定量机是否填满。

### Example — adhesive_dispensing

在Siasun机器人涂胶工作站上，操作员需为1号定量胶机配置工艺参数（程序号1、参数号1），完成一次从冲锋位出发、沿P4→P1轨迹涂胶、再返回冲锋位的完整循环。

**Environment and prerequisites:** 前置条件：胶机加热器与泵已通电且无故障（IN#72/IN#68/IN#69/IN#66均为ON）；系统接通OT#46与运行许可OT#40已置ON；程序号I492=1、参数号I493=1已通过OG#1/OG#2下发并获IG#1/IG#2确认；涂胶准备就绪IN#48=ON。变量：SysNum=1, Gun=1, ProgNo=1, ParaNo=1, Purge=Yes, Fill=Yes, VL=300, ACC=100, CNT=10, ET=1, CALIB=1, TIME=20。点位：PR102为冲锋位，P4为涂胶起点，P1为涂胶终点。

**Expected result:** 机器人从冲锋位PR102运动至涂胶起点P4，再沿P1方向以300mm/s线性运动；到达P1时1号胶枪提前20ms开启出胶，运动结束后20ms关闭胶枪；随后机器人返回P4和冲锋位PR102，胶机执行测量与填充检查，完成一次涂胶循环。

**Manual:** SX322012《新松机器人涂胶应用操作手册》(A-1) V5.0 [E1] [E2] [E3] [E5] [E7]

```text
SET UF #0
SET TF #1
PAYLOAD #1
MOVJ PR102 VJ=30.0 ACC=30 CNT=0
A08_GLUE GlueSetSysNum SysNum=1
A08_GLUE GlueInit Gun=1 ProgNo=1 ParaNo=1 Purge=Yes Fill=Yes
MOVL P4 VL=1000.0 ACC=100 CNT=30 ET=1
GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEON GUN=1 CALIB=1 PARANo=1 TIME=20
GLUEL P1 VL=300 ACC=100 CNT=10 ET=1 GLUEOFF TIME=20
MOVL P4 VL=1000.0 ACC=100 CNT=30 ET=1
A08_GLUE GlueEnd Gun=1 Fill=No
A08_GLUE GlueFillCheck Gun=1 Check
MOVJ PR102 VJ=30.0 ACC=30 CNT=0
RET
END
```

## [2/17] category_arc_welding

如何使用ARCON开始弧焊、沿焊缝运动并用ARCOFF结束焊接？请说明起弧文件与焊接文件的区别。

### Retrieved content summary

- ARCON #1 #2：指定起弧与焊接文件，其中文件#1为起弧文件，文件#2为焊接文件。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 12682–12685; corpus 4.0, revision V5.0 / B-5; object `76dbdbbe2784fa3f5ff11ade3d77a97bceb732f77591ccbf2d15f5589cf8132d`.

- ARCON [1] [1]：以起弧文件1起弧，焊接文件1焊接。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 2781–2784; corpus 5.0, revision V5.0 / C-1; object `215eaf15ab496c036b2ceb8d3c2651de78dc1e042c8d2e2f8aaf2b0af27133aa`.

- 指令 `ARCON #1 #2`：参数 #1 指定起弧文件，参数 #2 指定焊接文件。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 4631–4634; corpus 4.0, revision V5.0 / B-5; object `01ba11f79a0af26702faff35a502f50807d0a43a13a2eabbf77776e1d8b55bbe`.

- ARCON #1 #2 参数定义中，文件#1指定为起弧文件，文件#2指定为焊接文件。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 13848–13851; corpus 4.0, revision V5.0 / B-5; object `af5b6f3473b353ae04f105b8a3d8aeed2ff3eb9c3ffbfb9effd6df5036e559c9`.

- **指令参数 `ARCON #1 #2` 说明：#1 指定起弧文件1，#2 指定焊接文件2。**
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 8797–8800; corpus 4.0, revision V5.0 / B-5; object `994fc47187cd47a831e1c1e76006557858e1705a298d2ded314db327701378db`.

- ARCON 指令用于执行起弧操作，需同时指定起弧文件号 [1] 和焊接文件号 [1]。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(C-1)/SX322003《新松机器人弧焊应用操作手册》(C-1).md`, lines 5797–5800; corpus 5.0, revision V5.0 / C-1; object `a578110d80af410d7b4f99e16d3ec5d518042d738c3ec0d980c8c2c45e3bcd2c`.

- 指令 `ARCON #1 #2`：文件#1 为起弧文件，文件 2 为焊接文件。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 8960–8963; corpus 4.0, revision V5.0 / B-5; object `7c364c3924f7ca3c3771092d2cd89e29c28c2a44b98ddfd5005089006f18ca7c`.

- ARCON #1 #2：起弧/文件#1为起弧文件；文件#2为焊接文件
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322003《新松机器人弧焊应用操作手册》(B-5)/SX322003《新松机器人弧焊应用操作手册》(B-5).md`, lines 11890–11893; corpus 4.0, revision V5.0 / B-5; object `0b17944df06d5eb33366b76060f2a6775cca0ded889320ed03f08cc1429bc785`.

...

### Coding or diagnostic answer

FL_GETV 用于读取编码器传送带的实时速度。格式为 FL_GETV R<参数1> F<参数2>，其中参数1为用户实型R变量（如R10），用于存储读取到的速度值，单位为mm/s；参数2为通路选择，F1对应第1路编码器，F2对应第2路编码器。读取后，该R变量可直接作为FL_MOVL指令的VL参数使用，使机器人沿用户X方向以传送带实时速度进行跟随运动。FL_MOVL格式为 FL_MOVL VL=R<速度> T=R<距离> #<参数文件号>，其中VL为速度（mm/s），T为行进直线距离（mm），#1~#8为偏移参数文件号。

### Example — tracking

机器人需要跟随第一路传送带上的工件进行连续作业。程序先设置速度百分比为100，然后通过FL_GETV读取第一路编码器对应的传送带实时速度并存入R10，最后使用FL_MOVL以该速度执行跟随直线运动，行进距离由R1指定，偏移量由参数文件#1定义。

**Environment and prerequisites:** 前提条件：传送带编码器已连接并正常工作；用户坐标系#1和工具坐标系#1已设定；速度百分比必须设为100。变量：R10为速度存储变量（mm/s），R1为行进距离变量（mm），#1为偏移参数文件号（可选1~8）。初始值：R10由FL_GETV实时写入，R1由用户根据工件行程设定。

**Expected result:** FL_GETV执行后，R10中存储第一路编码器对应的传送带实时速度（单位mm/s）。随后FL_MOVL以R10的速度驱动机器人沿用户X方向运动R1指定的距离，YZ方向偏移由#1参数文件决定，实现与传送带同步的跟随作业。

**Manual:** SX322017《新松机器人跟随应用操作手册》(A-0) V5.0 / A-0 [E1] [E2] [E7] [E8]

```text
SET VEL=100
FL_GETV R10 F1
FL_MOVL VL=R10 T=R1 #1
```

## [17/17] category_vision

如何给相机发送拍照指令、获取测量结果并计算位姿偏移？请按同一手册版本说明视觉作业参数及指令调用顺序。

### Retrieved content summary

- 该表格定义了一个用于**获取相机测量结果**的功能接口，其调用格式为 `VSGPI PO#=（参数1） NO=（参数2）`。

具体参数配置如下：
*   **参数1**：对应 **视觉场景组**
*   **参数2**：对应 **视觉作业号**
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 426–443; corpus 4.0, revision V5.0 / A-1; object `67e7e8a38aa7ca8e2d355f9193ba99ae0dc2af2b885d53d581073c0e56c91b65`.

- 功能：给相机发送拍照指令。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 409–412; corpus 4.0, revision V5.0 / A-1; object `54a400cb18dd449c22f1ca9aad4d21df8006ffbf28f4646a7c7f25cf439e68be`.

- 该表格定义了功能指令 `VSGPI_N_P`，用于**获取相机测量结果**。

该指令包含四个参数（N, T, W, A），具体定义如下：
*   **参数1 (N)**：动作标志（0：视觉准备；1：触发测量）
*   **参数2 (T)**：功能模式（0：定位；1：打磨质量检测；2：凸起高度计算）
*   **参数3 (W)**：工件号（0-65535）
*   **参数4 (A)**：区域号（0-65535）；若在凸起高度计算模式下，兼作左右区分标志（0：左；1：右）
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 476–501; corpus 4.0, revision V5.0 / A-1; object `6235ecdd1ea97cbc2c9c60af869250f1009cf308a823fe9c12423de07734d845`.

- 功能：给相机发送拍照指令
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 451–454; corpus 4.0, revision V5.0 / A-1; object `da743cf2e7347e3af430fae472c4c7fcef43e6e31bae76fd3d63ca524a0e0340`.

- 该表格定义了用于**获取相机测量结果**的视觉系统指令格式 **`VSGPI PO#=（参数1） NO=（参数2）`**，并说明了两个关键参数的含义：**参数1** 指定**视觉场景组**，**参数2** 指定**视觉作业号**。
  Source: `/d/devel/rag/ribosome/assets/code-docs.5.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 426–443; corpus 5.0, revision V5.0 / A-1; object `cc120c4b8dea57918cc2045d25bc580c6d36493d3406281d96945072c4815808`.

- 功能：获取相机测量结果
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 480–483; corpus 4.0, revision V5.0 / A-1; object `5daf662f6b3aee7132e8e76afba8afdad7f91957f8e79f87cc6e43f04d11ed4d`.

- 功能：获取相机测量结果
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 430–433; corpus 4.0, revision V5.0 / A-1; object `80d1559762a13c0bf454438236ab4b82e849b51e203f5d1c387db60f2c337cec`.

- 该表定义了命令 `VSTG_N_P` 的参数结构及功能，用于**给相机发送拍照指令**。

*   **功能**：触发相机拍照。
*   **参数说明**：
    *   **参数1 (N)**：动作标志，0代表视觉准备测量，1代表触发视觉测量（范围：0-255）。
    *   **参数2 (T)**：功能模式选择，0代表定位，1代表打磨质量检测，2代表凸起高度计算（范围：0-255）。
    *   **参数3 (W)**：工件号（范围：0-65535）。
    *   **参数4 (A)**：区域号（范围：0-65535）；当参数2为2（凸起高度计算）时，该参数作为左右区分标志，0代表左，1代表右。
  Source: `/d/devel/rag/ribosome/assets/code-docs.4.0/.md/06 产品手册/软件类/手册/V5.0版本/SX322005《新松机器人视觉应用操作手册》（A-1)/SX322005《新松机器人视觉应用操作手册》（A-1).md`, lines 447–472; corpus 4.0, revision V5.0 / A-1; object `2d98089353bcf99359709b0a39395aa638658e54e6a466ff0e79fbc5cfeeba9f`.

### Coding or diagnostic answer

根据《新松机器人视觉应用操作手册》（A-1）V5.0版本，视觉作业中给相机发送拍照指令并获取测量结果的调用顺序为：首先使用 VSTG_N_P 指令以 N=0（视觉准备测量）进行初始化，然后以 N=1（触发视觉测量）正式触发拍照，最后使用 VSGPI_N_P 指令获取相机测量结果用于计算位姿偏移。VSTG_N_P 的参数包括：N（动作标志，0=准备/1=触发）、T（功能类型，0=定位/1=打磨质量检测/2=凸起高度计算）、W（工件号，0-65535）、A（区域号，0-65535，凸起高度计算时0=左/1=右）。VSGPI_N_P 参数结构相同，用于读取对应测量结果。

### Example — vision

机器人在装配工位上需要利用视觉系统对工件进行定位测量：先准备视觉测量环境，再触发相机拍照，最后读取测量结果以计算工件位姿偏移量并修正机器人末端轨迹。

**Environment and prerequisites:** 视觉场景已配置完成；视觉场景组编号为 1（占位符）；视觉作业号为 1（占位符）；功能类型 T=0 表示调用定位功能；工件号 W=1；区域号 A=0；动作标志 N=0 为视觉准备测量，N=1 为触发视觉测量。

**Expected result:** 第一条指令完成视觉测量准备；第二条指令触发相机拍照并执行定位测量；第三条指令返回测量结果（含位姿偏移数据），供后续运动补偿使用。

**Manual:** SX322005《新松机器人视觉应用操作手册》（A-1） V5.0 / A-1 [E4] [E8] [E3] [E6]

```text
VSTG_N_P N=0 T=0 W=1 A=0
VSTG_N_P N=1 T=0 W=1 A=0
VSGPI_N_P N=1 T=0 W=1 A=0
```

,case_id,category,hits,context_chars,omissions,missing_manuals,example_status,error
0,category_adhesive_dispensing,adhesive_dispensing,8,12971,0,0,included,
1,category_arc_welding,arc_welding,8,17215,0,0,included,
2,category_fds,fds,8,6403,0,0,included,
3,category_grinding,grinding,8,10491,0,0,unsupported,
4,category_laser_welding,laser_welding,8,11921,0,0,included,
5,category_load_analysis,load_analysis,8,4460,0,0,unsupported,
6,category_material_handling,material_handling,8,9870,0,0,included,
7,category_offline_interface,offline_interface,8,18685,0,0,unsupported,
8,category_palletizing,palletizing,8,8970,0,0,unsupported,
9,category_pneumatic_spot_welding,pneumatic_spot_welding,8,9161,0,0,included,


## Cleanup
Run after the selected category tests, or after interruption.


In [ ]:
# | notest
try:
    if 'app' in globals(): app.close()
finally:
    for client_name in ('embedding_client', 'generation_client'):
        if client_name in globals(): await globals()[client_name]._client.aclose()